In [ ]:
import pandas as pd
import re

# 1. Load the original dataset
print("Loading dataset...")
df = pd.read_csv('/content/Train.csv')
text_column = 'Data'

# Drop any completely empty rows
df = df.dropna(subset=[text_column])

# --- ADVANCED CLEANING (From your reference code) ---

# 2. Remove exact duplicates (Don't test the same sentence twice)
df = df.drop_duplicates(subset=[text_column])

# 3. Strip whitespace and newline characters from the edges
df[text_column] = df[text_column].apply(lambda x: str(x).strip(" \n\r\t"))

# 4. Remove rows containing URLs ("http")
df = df[~df[text_column].str.contains("http", na=False)]

# 5. Filter out rows with English letters (Keep only pure Bangla)
df = df[df[text_column].apply(lambda x: not bool(re.search(r'[A-Za-z]', str(x))))]


# --- THESIS SPECIFIC CLEANING ---

# 6. Define and remove target Responsibility Words
target_words = [
    "দায়িত্বশীল", "সতর্ক", "পরিণত", "পরিশ্রমী",
    "বেপরোয়া", "অবহেলা", "অলস", "আবেগপ্রবণ"
]

def contains_target_word(text):
    text_str = str(text)
    for word in target_words:
        if word in text_str:
            return True
    return False

# Drop rows that contain the target words
mask = df[text_column].apply(contains_target_word)
clean_df = df[~mask]

# 7. Filter by Word Length (Between 5 and 20 words is perfect for LLMs)
clean_df = clean_df[clean_df[text_column].apply(lambda x: 5 <= len(str(x).split()) <= 20)]

# 8. Sample 1,000 random rows for your LLM Experiment
sample_size = min(1000, len(clean_df))
final_df = clean_df.sample(n=sample_size, random_state=42)

# --- FORMATTING THE FINAL DATASET ---

# 9. Keep only the text column and rename it
final_df = final_df[[text_column]].reset_index(drop=True)
final_df.rename(columns={text_column: 'Comment_Text'}, inplace=True)


# 11. Save the final prepared dataset!
final_df.to_csv('Prepared_Thesis_Data_v2.csv', index=False)

print("\n--- PREPROCESSING COMPLETE ---")
print(f"Original rows: {len(df)}")
print(f"Clean rows available: {len(clean_df)}")
print(f"Final rows saved for testing: {len(final_df)}")

# Display the first 5 rows
final_df.head()

Loading dataset...

--- PREPROCESSING COMPLETE ---
Original rows: 12033
Clean rows available: 8998
Final rows saved for testing: 1000


,Comment_Text
0,মাদারচুদ চ্যানেল । আর কি কোন নিউজ নাই । এটাও ন...
1,কাচ্চি ভাই রেষ্টুরেন্টে গত বৃহষ্পতিবার দুপরে খ...
2,এক মাস সেহেরি খাইয়া রোজা রাহা সোজা
3,গালি বয় গানে বাংলাদেশের আসল চিত্র ভেসে আসে
4,সত্যের সন্ধানে নির্ভীক সাংবাদিকতাকে স্যালুট । ...


In [ ]:
# Save the processed DataFrame to a CSV file
final_df.to_csv("final_dataset.csv", index=False)

print("✅ Processed data saved as final_dataset.csv")

✅ Processed data saved as final_dataset.csv
